# Step 1: Reproduce idiom steering (IdioSteer)

Goal: build (or reuse, if already saved) the idiom-derived mean-difference steering vector from IdioLink and
confirm the core Steering Idioms result on `data/idiom_eval_benchmark_draft.csv`,
where steering should push continuations from figurative toward literal readings
while preserving fluency.

All reusable logic (model loading, activation collection, the steering hook,
the resumable eval loop, the judge) lives in `src/steering_pipeline.py`. This
notebook is a thin runner over it. GPU runtime required. The generation model
(Llama-3.2-3B) and the judge (Qwen3-14B, 4-bit) run sequentially, not
concurrently, so a free Colab T4 (~15GB usable VRAM) should be enough.

## Setup: clone repo, install deps, import the shared pipeline

In [ ]:
import os

REPO_DIR = "beyond-idiom-steering"
if not os.path.isdir(REPO_DIR):
    # Running fresh in Colab: clone the repo. If you already have the repo
    # mounted (e.g. via Drive or `%cd`), skip this cell and just make sure
    # your working directory is the repo root.
    !git clone https://github.com/Itamarvs/beyond-idiom-steering.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, ".")


In [ ]:
from src.steering_pipeline import *
import pandas as pd


## Mount Google Drive now

Do this first, not right before the judge step: mounting pops an interactive auth modal, and you don't want the whole sweep sitting blocked on a click hours into an unattended run. Results get written directly to Drive later (see the judge-labeling section) so progress survives a full Colab VM disconnect, not just a process crash.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/beyond-idiom-steering-results"
os.makedirs(DRIVE_DIR, exist_ok=True)
local_path = "results/labeled_idiom_results.csv"
drive_path = os.path.join(DRIVE_DIR, "labeled_idiom_results.csv")
if os.path.isfile(local_path):
    shutil.copy(local_path, drive_path)
    print(f"Copied {local_path} -> {drive_path}")
else:
    print("Nothing labeled yet, starting fresh on Drive.")

## 1. Load the base generation model

In [ ]:
model, tokenizer, device = load_generation_model()
print("device:", device, "| model:", MODEL_NAME)


## 2. Load or reuse the steering vector

The vector depends only on IdioLink + the model/layer config; none of
which change when the eval benchmark does. By default this loads the
already-saved `results/steering_vector_llama3.2-3b.pkl` and skips straight
to the sanity check, instead of re-downloading IdioLink and re-running the
activation-collection loop over its ~1,683 sentences. Set `REBUILD_VECTOR =
True` below only if you've changed the IdioLink filtering logic, the model,
or `SOURCE_LAYER`.

In [ ]:
REBUILD_VECTOR = False
VECTOR_PATH = "results/steering_vector_llama3.2-3b.pkl"

if not REBUILD_VECTOR and os.path.isfile(VECTOR_PATH):
    vec = load_steering_vector(VECTOR_PATH)
    v_md, s_md = vec["v_md"], vec["s_md"]
    assert vec["model"] == MODEL_NAME, (
        f"Saved vector was built on {vec['model']}, but MODEL_NAME is {MODEL_NAME}."
    )
    print(f"Loaded existing vector ({VECTOR_PATH}): "
          f"source_layer={vec['source_layer']}, s_md={s_md:.4f}")
else:
    print("Rebuilding steering vector from IdioLink "
          f"({'REBUILD_VECTOR=True' if REBUILD_VECTOR else f'no saved vector at {VECTOR_PATH}'})...")

    # Combines the `indexes` and `queries` configs (the paper's stated 80
    # sentences/idiom pool only comes out right with both), keeps
    # idiomatic/literal rows, and applies the paper's exact-surface-form filter.
    pool_df = load_idiolink_pool()
    print("idioms:", pool_df["idiom"].nunique(), "| pool size:", len(pool_df))
    print(pool_df["label"].value_counts())

    v_md, s_md = build_steering_vector(pool_df, model, tokenizer, device, source_layer=SOURCE_LAYER)
    print("raw mean-difference norm (s_md):", s_md)
    print("unit vector shape:", v_md.shape)

    save_steering_vector(VECTOR_PATH, v_md, s_md, source_layer=SOURCE_LAYER, model_name=MODEL_NAME)
    print(f"saved {VECTOR_PATH}")

## 3. Sanity check on a few idiom prefixes

In [ ]:
eval_df = pd.read_csv("data/idiom_eval_benchmark_draft.csv")

sanity_idioms = ["break the ice", "spill the beans", "see red"]
sanity_rows = eval_df[eval_df["idiom"].isin(sanity_idioms) & (eval_df["variant_id"] == 1)]

for _, row in sanity_rows.iterrows():
    prefix = row["prefix"]
    print("=" * 80)
    print(f"IDIOM: {row['idiom']}")
    unsteered = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=0.0, n_samples=1)[0]
    print(f"UNSTEERED : {prefix}{unsteered}")
    steered_lit = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=-4.78, n_samples=1)[0]
    print(f"STEERED(-4.78, literal push): {prefix}{steered_lit}")
    print()


## 4. Full evaluation sweep (resumable; safe to rerun after a disconnect)

In [ ]:
idiom_results = run_generation_eval(
    model, tokenizer, device, eval_df, v_md, s_md,
    out_csv="results/raw_idiom_results.csv",
)
print(f"total rows: {len(idiom_results)}")


## 5. Free the generation model before loading the judge

Tight on a single T4: the 3B generation model plus a 4-bit-quantized 14B
judge should both fit (~6GB + ~9GB), but only if the generation model isn't
still resident.

In [ ]:
del model
torch.cuda.empty_cache()
print("Generation model freed.")


## 6. LLM-as-judge labeling (Qwen3-14B, 4-bit)

The Steering Idioms paper (Appendix J) used **Gemma-4-31B-it** as its judge,
accessed via the OpenRouter API, not self-hosted (Appendix Q), selected by
benchmarking several candidates (GPT-4o, Claude Sonnet 3.5, Llama-3.3-70B-Instruct,
Qwen3, ...) against a 280-item human-annotated gold set: 90.0% accuracy,
Cohen's kappa=0.821 vs. gold (human-human kappa=0.867). Only the winning
judge's score is reported; Qwen3's standing among the candidates isn't
disclosed.

A 31B judge doesn't fit a free-tier Colab GPU, so this uses **Qwen3-14B in
4-bit**, in the paper's candidate pool (so evidence-adjacent) but not the
winner, and with no calibration evidence of its own yet. Treat its labels as
provisional until the small human-calibration pass (see README) is run. It
replaces the original Qwen2.5-3B-Instruct judge (see
`results/labeled_idiom_results_qwen3b_baseline.csv`), which produced a flat
literal rate across the whole alpha grid (~0.30-0.41, no clear steering
effect), unlike the paper's own reported result at this *exact* calibrated
config (`L_s=14, L_i=2, alpha_factor` grid), which shows baseline 0.12 -> 0.49
at alpha_factor=-4.78. That mismatch, at an otherwise identical config,
points at the judge as the likely weak link, not the steering vector. This
cell is the test of that hypothesis.


**Speed**: `run_judge_eval` batches judge calls (`batch_size=16` below) instead
of one `generate()` call per row; unbatched labeling on a T4 with this
4-bit 14B judge runs ~25-30s/item, which doesn't fit a session for anything
but the smallest sweeps. It flushes to disk every `checkpoint_every` batches
and is resumable (reruns skip already-labeled rows), so a crash loses at
most a few in-flight items.

**Resiliency**: that protects against the *process* crashing, but a full
Colab VM disconnect/recycle takes local disk with it. Drive was already mounted at the top of this notebook, so that isn't
a blocker here: `run_judge_eval` below writes `labeled_idiom_results.csv` there directly, so
progress survives a disconnect without any manual copying.

In [ ]:
judge_model, judge_tokenizer, device = load_judge_model()


In [ ]:
labeled_df = run_judge_eval(
    judge_model, judge_tokenizer, device,
    in_csv="results/raw_idiom_results.csv",
    out_csv=drive_path,
    expr_col="idiom",
    batch_size=16,
    checkpoint_every=5,
)
summarize_labels(labeled_df, group_cols=["alpha_factor"])

## 7. Commit results (optional; run manually, review `git status` first)

Only run this if you actually want to push from the Colab session. Safer
default: download `results/*.csv` and the `.pkl` and commit locally.

In [ ]:
# !git add results/
# !git commit -m "Step 1: steering vector + idiom eval results"
# !git push
